<a href="https://colab.research.google.com/github/pooriya-zakerian/quasar/blob/IlCallo-patch-v1/chat_with_data_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import data

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import pandas as pd

In [3]:
data = pd.read_csv("train.csv")

In [4]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
##from google.colab import drive
##drive.mount('/content/drive')

In [6]:
len(data), data.dtypes

(891,
 PassengerId      int64
 Survived         int64
 Pclass           int64
 Name            object
 Sex             object
 Age            float64
 SibSp            int64
 Parch            int64
 Ticket          object
 Fare           float64
 Cabin           object
 Embarked        object
 dtype: object)

## Convert DataFrame to SQLite Database

In [7]:
from sqlite3 import connect

In [8]:
con = connect('titanic.db')
data.to_sql('Passenger Data', con, if_exists = 'replace')

891

In [9]:
!pip install langchain-community
from langchain_community.utilities import SQLDatabase

In [10]:
db = SQLDatabase.from_uri("sqlite:///titanic.db", sample_rows_in_table_info = 3)
print(db.table_info)


CREATE TABLE "Passenger Data" (
	"index" INTEGER, 
	"PassengerId" INTEGER, 
	"Survived" INTEGER, 
	"Pclass" INTEGER, 
	"Name" TEXT, 
	"Sex" TEXT, 
	"Age" REAL, 
	"SibSp" INTEGER, 
	"Parch" INTEGER, 
	"Ticket" TEXT, 
	"Fare" REAL, 
	"Cabin" TEXT, 
	"Embarked" TEXT
)

/*
3 rows from Passenger Data table:
index	PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
0	1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
1	2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
2	3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S
*/


## Initialize Language Model from OpenAI

In [11]:
!pip install -q langchain langchain-nvidia-ai-endpoints gradio


In [27]:

import os
os.environ["NVIDIA_API_KEY"] = "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"
os.environ["LANGCHAIN API_KEY"] = "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"
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "langsmith-onboarding"


from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [28]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os

In [14]:
## Useful utility method for printing intermediate states
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State: "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def PPrint(preface="State: "):
    def print_and_return(x, preface=""):
        pprint(preface, x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

In [29]:
from langchain_core.runnables import RunnableLambda
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Union
from operator import itemgetter

In [30]:


instruct_llm = ChatNVIDIA(model="mistralai/mistral-7b-instruct-v0.2")

In [31]:
instruct_llm.config_schema

<bound method Runnable.config_schema of ChatNVIDIA(base_url='https://integrate.api.nvidia.com/v1', model='mistralai/mistral-7b-instruct-v0.2')>

## Using SQLDatabase Chain

In [32]:
!pip install langchain-experimental
from langchain_experimental.sql.base import SQLDatabaseChain

In [33]:
sql_db_chain = SQLDatabaseChain.from_llm(llm=instruct_llm, db=db, verbose=True)

In [34]:
sql_db_chain.invoke("How many passengers were there?")



> Entering new SQLDatabaseChain chain...
How many passengers were there?
SQLQuery:SQLQuery: SELECT COUNT(*) FROM "Passenger Data";
SQLResult:
SQLResult: [(891,)]
Answer:There were 891 passengers in total.
> Finished chain.


{'query': 'How many passengers were there?',
 'result': 'There were 891 passengers in total.'}

In [35]:


sql_db_chain.invoke("How does survival rate varies on the basis of passenger classes?")



> Entering new SQLDatabaseChain chain...
How does survival rate varies on the basis of passenger classes?
SQLQuery:SELECT `Pclass`, AVG("Survived") as "Average Survival Rate"
FROM "Passenger Data"
GROUP BY "Pclass"
ORDER BY "Pclass"
LIMIT 5;

SQLResult:
SQLResult: [(1, 0.6296296296296297), (2, 0.47282608695652173), (3, 0.24236252545824846)]
Answer:The survival rate for each passenger class is as follows:
Class 1: 0.6296
Class 2: 0.4728
Class 3: 0.2424

Therefore, the survival rate tends to decrease as the passenger class goes from first to third.
> Finished chain.


{'query': 'How does survival rate varies on the basis of passenger classes?',
 'result': 'The survival rate for each passenger class is as follows:\nClass 1: 0.6296\nClass 2: 0.4728\nClass 3: 0.2424\n\nTherefore, the survival rate tends to decrease as the passenger class goes from first to third.'}

## Custom Prompt

In [ ]:


sql_db_chain = SQLDatabaseChain.from_llm(llm = instruct_llm, db = db, verbose = True, prompt = PROMPT)
sql_db_chain.invoke("Provide the gender-wise survival rate along with age bins of 20.")

In [63]:
connection = connect("titanic.db")

In [64]:
curr = connection.execute('''
SELECT "Sex",
    CASE
        WHEN "Age" < 20 THEN '0-19'
        WHEN "Age" >= 20 AND "Age" < 40 THEN '20-39'
        WHEN "Age" >= 40 AND "Age" < 60 THEN '40-59'
        ELSE '60+'
    END AS "Age_Bin",
    AVG("Survived") AS "Survival_Rate"
FROM "Passenger Data"
GROUP BY "Sex", "Age_Bin"
ORDER BY "Sex", "Age_Bin";
''')

In [39]:
print(curr.fetchall())

[('female', '0-19', 0.7066666666666667), ('female', '20-39', 0.7727272727272727), ('female', '40-59', 0.76), ('female', '60+', 0.7017543859649122), ('male', '0-19', 0.29213483146067415), ('male', '20-39', 0.18823529411764706), ('male', '40-59', 0.1839080459770115), ('male', '60+', 0.13013698630136986)]


In [65]:
curr.close()
connection.close()

In [70]:
prompt_template = '''
You are an AI agent that answers questions by:
1. Generating a syntactically correct SQLite query.
2. Running the query and examining the result.
3. Explaining the result in plain English as a human-readable answer.

Use the following format:

Question: <User's question>
SQLQuery: <Valid SQLite query>
SQLResult: <Output of the query as a list of tuples>
Answer: <Interpretation of the result that answers the user's question>

### Guidelines:
- Use only the tables and columns given.
- DO NOT use SELECT *.
- Always wrap column names in double quotes.
- If the question includes date/time reference like "today", use `date('now')`.
- Use LIMIT 10 unless the user asks for more.
- Explain the result logically and informatively.

### Example:
Question: Provide the gender-wise survival rate along with age bins of 20.
SQLQuery: SELECT "Sex",
       CASE
           WHEN "Age" < 20 THEN '0-19'
           WHEN "Age" >= 20 AND "Age" < 40 THEN '20-39'
           WHEN "Age" >= 40 AND "Age" < 60 THEN '40-59'
           ELSE '60+'
       END AS "AgeBin",
       AVG("Survived") AS "SurvivalRate"
FROM "Passenger Data"
GROUP BY "Sex", "AgeBin"
ORDER BY "Sex", "AgeBin"
LIMIT 10;
SQLResult: [('female', '0-19', 0.70), ('female', '20-39', 0.77), ('female', '40-59', 0.76), ('female', '60+', 0.70), ('male', '0-19', 0.29), ('male', '20-39', 0.18), ('male', '40-59', 0.18), ('male', '60+', 0.13)]
Answer: Female passengers had significantly higher survival rates than male passengers across all age bins. Females aged 20–39 had the highest rate (~77%), while males aged 60+ had the lowest (~13%).

Only use the following table:
CREATE TABLE "Passenger Data" (
    "index" INTEGER,
    "PassengerId" INTEGER,
    "Survived" INTEGER,
    "Pclass" INTEGER,
    "Name" TEXT,
    "Sex" TEXT,
    "Age" REAL,
    "SibSp" INTEGER,
    "Parch" INTEGER,
    "Ticket" TEXT,
    "Fare" REAL,
    "Cabin" TEXT,
    "Embarked" TEXT
)

Sample rows:
index	PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked
0	1	0	3	Braund, Mr. Owen Harris	male	22.0	1	0	A/5 21171	7.25	None	S
1	2	1	1	Cumings, Mrs. John Bradley (Florence Briggs Thayer)	female	38.0	1	0	PC 17599	71.2833	C85	C
2	3	1	3	Heikkinen, Miss. Laina	female	26.0	0	0	STON/O2. 3101282	7.925	None	S

Question: {input}
'''


In [71]:
from langchain_core.prompts import PromptTemplate
PROMPT = PromptTemplate.from_template(prompt_template, variables = ['input'])

In [72]:
sql_db_chain = SQLDatabaseChain.from_llm(llm = instruct_llm, db = db, prompt = PROMPT)

In [75]:
result = sql_db_chain.invoke("Provide the gender-wise survival rate along with age bins of 20")
result['result']

"Female passengers had a higher survival rate than male passengers across all age bins. Females in the age group '20-39' had the highest survival rate of approximately 77% on average. The survival rate for males in the age group '60+' was around 13%. For other age groups, females had a survival rate around 70-77%, and males had a survival rate around 18-29%."

## Create a SQL Query Chain

In [76]:
from langchain.chains import create_sql_query_chain

In [77]:
sql_chain = create_sql_query_chain(instruct_llm, db)

In [78]:
sql_chain

RunnableAssign(mapper={
  input: RunnableLambda(...),
  table_info: RunnableLambda(...)
})
| RunnableLambda(lambda x: {k: v for k, v in x.items() if k not in ('question', 'table_names_to_use')})
| PromptTemplate(input_variables=['input', 'table_info'], input_types={}, partial_variables={'top_k': '5'}, template='You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.\nUnless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.\nNever query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.\nPay attention to use only the column names you can see in the

In [79]:
response = sql_chain.invoke({"question": "How many passengers were there?"})
response

'SQLQuery: SELECT COUNT(*) FROM "Passenger Data";\nSQLResult:'

## Run SQL Query on your database

In [82]:
!pip install -U langchain-community

In [80]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

In [83]:
db_execution = QuerySQLDataBaseTool(db = db)
execution_chain = sql_chain | db_execution

In [98]:
response = execution_chain.invoke({"question": "How many passengers were there?"})
response

'Error: (sqlite3.OperationalError) near "SQLQuery": syntax error\n[SQL: SQLQuery: SELECT COUNT(*) FROM "Passenger Data";\nSQLResult:]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)'

## Summarize the final result

In [85]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [146]:
from langchain_core.prompts import PromptTemplate

template = '''
Here's the user's question, the SQL query used to answer it, and the result of the query.
Just based on the result, give a short and natural explanation in plain English. Be brief, casual, and to the point.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer:
'''


prompt = PromptTemplate.from_template(template)

In [138]:
def clean_sql_query(query):
    # Remove "SQLResult:" and any trailing whitespace
    cleaned_query = query.split("SQLResult:")[0].strip()
    return cleaned_query

def format_result(result):
    if isinstance(result, list):
        return str([
            {"Pclass": row[0], "AverageFare": round(row[1], 2)}
            for row in result
        ])
    return str(result)


In [147]:
chain = (
    RunnablePassthrough
    .assign(query=sql_chain) # Changed 'sql_query' to 'query'
    .assign(cleaned_query=itemgetter("query") | RunnableLambda(clean_sql_query)) # Changed itemgetter to use 'query'
    .assign(raw_result=itemgetter("cleaned_query") | db_execution)
    .assign(result=itemgetter("raw_result") | RunnableLambda(format_result))
    | prompt
    | instruct_llm
    | StrOutputParser()
)

In [141]:
ch = RunnablePassthrough.assign(query = sql_chain)

In [132]:
ch.invoke({"question": "How does the prices varies with respect to passenger classes?"})

{'question': 'How does the prices varies with respect to passenger classes?',
 'query': 'SELECT "Pclass", AVG("Fare")\nFROM "Passenger Data"\nGROUP BY "Pclass"\nORDER BY "Pclass"\nLIMIT 5;\n\nSQLResult:'}

In [133]:

ch2 = RunnablePassthrough.assign(query=sql_chain).assign(cleaned_query=itemgetter("query") | RunnableLambda(clean_sql_query)).assign(result=itemgetter("cleaned_query") | db_execution)

In [134]:
ch2.invoke({"question": "How does the prices varies with respect to passenger classes?"})

{'question': 'How does the prices varies with respect to passenger classes?',
 'query': 'SELECT "Pclass", AVG("Fare")\nFROM "Passenger Data"\nGROUP BY "Pclass"\nORDER BY "Pclass"\nLIMIT 5;\n\nSQLResult:',
 'cleaned_query': 'SELECT "Pclass", AVG("Fare")\nFROM "Passenger Data"\nGROUP BY "Pclass"\nORDER BY "Pclass"\nLIMIT 5;',
 'result': '[(1, 84.15468749999992), (2, 20.66218315217391), (3, 13.675550101832997)]'}

In [149]:
chain.invoke({"question": "How does the prices varies with respect to passenger classes?"})

' The SQL query you provided calculates the average fare for each passenger class (represented by "Pclass") and returns the top 5 classes ordered by their average fares. The result shows that the average fare for the first class is around $84, the second class is roughly $21, and the third class is approximately $14. In simple terms, the prices vary significantly among different passenger classes, with the first class being the most expensive and the third class being the least expensive.'

In [105]:
#//chat
prompt_text = prompt_template.format(input="How does the prices varies with respect to passenger classes?")
response = instruct_llm.invoke(prompt_text)
print(response)


content=' SQLQuery: SELECT "Pclass", AVG("Fare") AS "AveragePrice" FROM "Passenger Data" GROUP BY "Pclass" ORDER BY "Pclass" LIMIT 3;\nSQLResult: [(1, 7.32325), (2, 14.1468), (3, 29.0925)]\nAnswer: The average fare prices generally increase with passenger class. First class passengers paid the highest average price (around 29 dollars), while third class passengers paid the lowest average price (around 7 dollars). Second class passengers paid an average fare of around 14 dollars.' additional_kwargs={} response_metadata={'role': 'assistant', 'content': ' SQLQuery: SELECT "Pclass", AVG("Fare") AS "AveragePrice" FROM "Passenger Data" GROUP BY "Pclass" ORDER BY "Pclass" LIMIT 3;\nSQLResult: [(1, 7.32325), (2, 14.1468), (3, 29.0925)]\nAnswer: The average fare prices generally increase with passenger class. First class passengers paid the highest average price (around 29 dollars), while third class passengers paid the lowest average price (around 7 dollars). Second class passengers paid an av

## Showcasing in UI using Gradio

In [113]:
import gradio as gr

In [114]:
template = '''
Here's the user's question, the SQL query used to answer it, and the result of the query.
Just based on the result, give a short and natural explanation in plain English. Be brief, casual, and to the point.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer:
'''
prompt = PromptTemplate.from_template(template)

In [125]:
def create_chain(question):

    db = SQLDatabase.from_uri("sqlite:///titanic.db", sample_rows_in_table_info = 3)

    sql_chain = create_sql_query_chain(instruct_llm, db)
    db_execution = QuerySQLDataBaseTool(db = db)
    output = prompt | instruct_llm | StrOutputParser()
    chain = (RunnablePassthrough.assign(sql_query = sql_chain).assign(result = itemgetter("sql_query") | db_execution) | output)

    return chain.invoke({"question": question})


In [126]:
def extract_data(user_message, history):
    question_with_history = ""
    for hist in history[-3:]:
        question_with_history += f"User: {hist[0]}\nAssistant: {hist[1]}\n"
    question_with_history += f"User: {user_message}\n"
    print("Input to LLM:\n", question_with_history)

    bot_message = create_chain(question_with_history)

    history += [[user_message, bot_message]]

    return bot_message, history

In [127]:
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label = "Chat with Data")
    msg = gr.Textbox(label = "Question", placeholder = "Enter your question here")
    clear = gr.Button("Clear")

    def user(user_message, history):
        bot_message, history = extract_data(user_message, history)
        print("LLM Response: ", bot_message)
        return "", history

    msg.submit(user, [msg, chatbot], [msg, chatbot], queue=False)
    clear.click(lambda: None, None, chatbot, queue=False)

demo.queue()
demo.launch(debug=True)

<ipython-input-127-de7b7ee08231>:2: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label = "Chat with Data")


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://94352a45702d551abc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Input to LLM:
 User: hi

LLM Response:   The user's question "hi" does not correspond to an SQL query or result. SQL is a programming language used for managing and manipulating data in relational databases, and it requires a specific syntax and structure, such as a SELECT statement, to retrieve information from a database. A simple "hi" from a user is not related to SQL or database queries. Therefore, there is no SQL query, result, or answer for this question.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://94352a45702d551abc.gradio.live
